# SARBLOH Prime: E008, autonomous Prime agent with pushed perception and agent-owned memory (17 dev games)

The agent is our Prime Agent port (`Sarbloh-Arc/prime/`, upstream kernel `Sarbloh-Arc/rlm/` unchanged) with the E008
toolset: `ipython` (read-only `scene`), `act` (actions + plan/hypotheses/findings/goal) and `recall`. After every act
the new state is pushed (change lines, objects, changed region, PNG). The model is Qwen3.8-27B-FP8 on vLLM 0.19.0:
profile `qwen_fp8_vision` (1 image per prompt), falling back to `qwen_fp8` (text only) if the image start fails.
See `experiments/E008_perception_memory/hypothesis.md`.

Attach: the competition, `banwait13/sarblohagent` (this repo, updated after the push),
`fumiyauchiyama/qwen3-8-27b-fp8-hf-snapshot` (official Qwen/Qwen3.8-27B-FP8), `banwait13/sarbloh-vllm-wheelhouse`
(vLLM), and the RTX Pro 6000. Internet **off**.

## 0. CONFIG: every knob is here (overrides of `prime/config.py`)

In [ ]:
import time
NOTEBOOK_START = time.time()

# E008 arm A. Arms B / C are config diffs: {"agent": {"perception": {"image": False}}} / {"agent": {"memory":
# {"lessons": False}}}; run them only if arm A clears >= 2 levels (hypothesis.md).
OVERRIDES = {
    "model": "qwen",                  # prime/llm/qwen.py: chain qwen_fp8_vision -> qwen_fp8
    "experiment": "E008_perception_memory",
    # the 17 dev games (held-out ar25 cd82 ft09 g50t m0r0 s5i5 sk48 vc33 are never run here)
    "games": ["bp35", "cn04", "dc22", "ka59", "lf52", "lp85", "ls20", "r11l", "re86", "sb26", "sc25", "sp80", "su15",
              "tn36", "tr87", "tu93", "wa30"],
    # 6 at a time: KV cache ~174k tokens (E007 FP8 at 65k) / ~30k per game context. UNCONFIRMED for E008's prompts.
    "concurrency": 6,
    "notebook_budget_s": 18000.0,     # 5 h: three waves of 6 games at up to 1.5 h each (UNCONFIRMED pace)
    "teardown_reserve_s": 300.0,
    "game_wall_s": 5400.0,
    "max_actions_per_game": 400,
    "record": True,                   # ARC SDK recording + agent step refs -> prime_run/trace.md
    "agent": {"toolset": "e008", "max_depth": 0},
    "vllm": {"profile_chain": None},  # None = the model spec's chain
}

# Competition submission (Phase B rerun on the hidden games): the gateway serves the games and scores them, so
# "games" is ignored. 12 h hard limit; ends at 11.5 h. Hidden game count is UNCONFIRMED. Save & Run (Phase A) keeps
# the dev config above.
import os
if os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}:
    OVERRIDES.update({
        "experiment": "SUB003_prime_e008",
        "notebook_budget_s": 41400.0,     # 11.5 h
        "teardown_reserve_s": 600.0,
        "game_wall_s": 10800.0,           # 3 h per game
        "max_actions_per_game": 2000,     # unfinished levels score 0 either way; more actions only add chances
        "record": False,
    })
AGENT_DATASET = "banwait13/sarblohagent"

## 1. ARC runtime (competition wheels, offline) and the agent source

In [ ]:
import os, subprocess, sys
from pathlib import Path

COMPETITION_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-warn-conflicts",
                       "--disable-pip-version-check", "--find-links", str(COMPETITION_DIR / "arc_agi_3_wheels"), "arc-agi"],
                      stdout=subprocess.DEVNULL)

owner, slug = AGENT_DATASET.split("/")
candidates = [Path("/kaggle/input/datasets") / owner / slug, Path("/kaggle/input") / slug]
AGENT_ROOT = next((p for p in candidates if (p / "Sarbloh-Arc" / "prime").exists()), None)
if AGENT_ROOT is None:
    hits = [p.parent.parent for p in Path("/kaggle/input").glob("**/Sarbloh-Arc/prime/run.py")]
    AGENT_ROOT = hits[0].parent if hits else None
if AGENT_ROOT is None:
    raise FileNotFoundError(f"Sarbloh-Arc not found: attach {AGENT_DATASET} (updated from the git repo)")
for p in (AGENT_ROOT / "Sarbloh-Arc", AGENT_ROOT):
    sys.path.insert(0, str(p))
print("agent root:", AGENT_ROOT)
print(subprocess.run(["ls", "-la", str(AGENT_ROOT / "Sarbloh-Arc")], capture_output=True, text=True).stdout)

## 2. Run: vLLM up (profile chain + smoke test), play the games, teardown

In [ ]:
from prime.run import main

SUMMARY = main(OVERRIDES, notebook_start=NOTEBOOK_START)

## 3. Summary

In [ ]:
import json
print(json.dumps(SUMMARY, indent=1, default=str))
print(f"notebook wall: {time.time() - NOTEBOOK_START:.0f}s")

## 4. Step trace: per game `steps.md` (turn by turn), `steps.jsonl`, SDK recording, SDK scorecard

In [ ]:
trace_md = next(Path("/kaggle/working").rglob("prime_run/trace.md"), None)
if trace_md is None:
    print("no trace.md (run crashed before results.json?)")
else:
    print(trace_md.read_text(encoding="utf-8"))
    run_dir = trace_md.parent
    for f in sorted(run_dir.rglob("*")):
        if f.is_file() and f.suffix in (".md", ".jsonl", ".json"):
            print(f"{f.stat().st_size:>10}  {f.relative_to(run_dir)}")